In [95]:
import os
import sys

# 1. Force the notebook to run from your project workspace root
project_root = r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression"
os.chdir(project_root)

# 2. Add the 'src' folder to Python's system path so it can see 'Linear_regression_01'
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print("Current Working Directory:", os.getcwd())
print("System path updated. Available modules:", os.listdir(src_path))

Current Working Directory: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression
System path updated. Available modules: ['Linear_regression_01', 'Linear_regression_01.egg-info']


In [96]:
import os
print(os.getcwd())

C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression


In [97]:
import sys
print(sys.path)

['C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire', '', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages', 'C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\linear_regression\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\pythonwin']


In [98]:
import os

print(os.path.exists("../src"))
print(os.path.exists("../src/Linear_regression_01"))

False
False


In [99]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(PROJECT_ROOT)
print(SRC_PATH)
print(sys.path[:3])

C:\Users\Greesha Vaishnavi\Desktop\dsprojects
C:\Users\Greesha Vaishnavi\Desktop\dsprojects\src
['C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs']


In [100]:
import box
print(box.__version__)

7.4.1


In [101]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir : Path
    STATUS_FILE : Path
    unzip_data_dir : Path
    transformed_train_path : Path
    transformed_test_path : Path
    preprocessor_path : Path

In [102]:
from Linear_regression_01.entity.config_entity import DataTransformationConfig
from Linear_regression_01.utils.common import read_yaml, create_directories
from Linear_regression_01.constant import CONFIG_FILE_PATH

In [103]:
# configuration manager

from Linear_regression_01.config.configuration import ConfigurationManager

config = read_yaml(CONFIG_FILE_PATH)

create_directories([config.data_transformation.root_dir])

data_transformation_config = DataTransformationConfig(
    root_dir=Path(config.data_transformation.root_dir),
    STATUS_FILE=Path(config.data_transformation.STATUS_FILE),
    unzip_data_dir=Path(config.data_transformation.unzip_data_dir),
    transformed_train_path=Path(config.data_transformation.transformed_train_path),
    transformed_test_path=Path(config.data_transformation.transformed_test_path),
    preprocessor_path=Path(config.data_transformation.preprocessor_path)
)



[2026-07-25 23:15:02,072: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-25 23:15:02,085: INFO: common: created directory at artifacts/data_transformation]


In [104]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from Linear_regression_01.entity.config_entity import DataTransformationConfig
from Linear_regression_01.logging import logger

In [105]:
def binary_encoding(X):

    X = X.copy()

    mapping = {
        "yes": 1,
        "no": 0
    }

    return X.replace(mapping)

In [106]:
# component

class DataTransformation:

    def __init__(self, config: DataTransformationConfig):
        self.config = config


    def initiate_data_transformation(self):

        logger.info("***** Data Transformation Started *****")

        # Check Validation Status


        with open(self.config.STATUS_FILE,"r") as f:

            status = f.read().split(" ")[-1]

        if status != "True":
            raise Exception("Data Validation Failed")

        logger.info("Validation Successful")


        # -----------------------------
        # Read Dataset
        # -----------------------------

        df = pd.read_csv(self.config.unzip_data_dir)

        logger.info(f"Dataset Shape : {df.shape}")


        # -----------------------------
        # Split Features and Target
        # -----------------------------

        X = df.drop(columns=["price"],axis=1)

        y = df["price"]


        # -----------------------------
        # Train Test Split
        # -----------------------------

        X_train,X_test,y_train,y_test = train_test_split(

            X,
            y,
            test_size=0.20,
            random_state=42

        )

        logger.info("Train Test Split Completed")


        # -----------------------------
        # Columns
        # -----------------------------

        binary_columns = [

            "mainroad",
            "guestroom",
            "basement",
            "hotwaterheating",
            "airconditioning",
            "prefarea"

        ]

        categorical_column = [

            "furnishingstatus"

        ]


        # -----------------------------
        # Pipelines
        # -----------------------------

        binary_pipeline = Pipeline(

            steps=[

                ("binary",FunctionTransformer(binary_encoding))

            ]

        )


        categorical_pipeline = Pipeline(

            steps=[

                ("onehot",OneHotEncoder(handle_unknown="ignore"))

            ]

        )


        # -----------------------------
        # Column Transformer
        # -----------------------------

        preprocessor = ColumnTransformer(

            transformers=[

                ("binary",binary_pipeline,binary_columns),

                ("categorical",categorical_pipeline,categorical_column)

            ],

            remainder="passthrough"

        )


        logger.info("Applying Preprocessing")


        X_train = preprocessor.fit_transform(X_train)

        X_test = preprocessor.transform(X_test)


        logger.info("Preprocessing Completed")
        # -----------------------------
        # Save Preprocessor
        # -----------------------------

        joblib.dump(preprocessor, self.config.preprocessor_path)

        logger.info("Preprocessor Saved")


        # -----------------------------
        # Combine Features and Target
        # -----------------------------

        train_arr = np.c_[X_train, np.array(y_train)]

        test_arr = np.c_[X_test, np.array(y_test)]


        # -----------------------------
        # Save Train and Test Arrays
        # -----------------------------

        np.save(self.config.transformed_train_path, train_arr)

        np.save(self.config.transformed_test_path, test_arr)

        logger.info("Transformed Data Saved")


        return (
        self.config.transformed_train_path,
        self.config.transformed_test_path,
        self.config.preprocessor_path





    )



In [107]:
# Pipeline

try:

    logger.info("*******************")
    logger.info("Data Transformation Stage Started")

    obj = DataTransformation(config=data_transformation_config)

    train_path, test_path, preprocessor_path = obj.initiate_data_transformation()

    logger.info("Data Transformation Stage Completed")

    print("Train Path :", train_path)
    print("Test Path :", test_path)
    print("Preprocessor Path :", preprocessor_path)

except Exception as e:

    logger.exception(e)

    raise e

[2026-07-25 23:15:13,829: INFO: 2953653647: *******************]
[2026-07-25 23:15:13,831: INFO: 2953653647: Data Transformation Stage Started]
[2026-07-25 23:15:13,837: INFO: 1045997381: ***** Data Transformation Started *****]
[2026-07-25 23:15:13,840: INFO: 1045997381: Validation Successful]
[2026-07-25 23:15:13,864: INFO: 1045997381: Dataset Shape : (545, 13)]
[2026-07-25 23:15:13,876: INFO: 1045997381: Train Test Split Completed]
[2026-07-25 23:15:13,879: INFO: 1045997381: Applying Preprocessing]
[2026-07-25 23:15:13,913: INFO: 1045997381: Preprocessing Completed]
[2026-07-25 23:15:13,919: INFO: 1045997381: Preprocessor Saved]
[2026-07-25 23:15:13,919: INFO: 1045997381: Transformed Data Saved]
[2026-07-25 23:15:13,919: INFO: 2953653647: Data Transformation Stage Completed]
Train Path : artifacts\data_transformation\train.npy
Test Path : artifacts\data_transformation\test.npy
Preprocessor Path : artifacts\data_transformation\preprocessor.pkl


C:\Windows\Temp\ipykernel_7792\287080829.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping)
C:\Windows\Temp\ipykernel_7792\287080829.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping)
